# Masterclass: Advanced RAG Retrieval Techniques
This comprehensive guide explores the full spectrum of retrieval techniques in Retrieval-Augmented Generation (RAG). It covers **Dense Vector Search**, **Maximal Marginal Relevance (MMR)**, **Score Thresholding**, **Mathematical Distance Metrics**, **Metadata Filtering (Pre vs. Post)**, **Hybrid Search (BM25 + Dense RRF)**, **Query Transformations (Rewriting, Expansion, Decomposition)**, and **Cross-Encoder Reranking**.

## 1. Setup, Imports, and API Configuration
Import core Python utilities, LangChain document loaders, text splitters, Google Gemini embeddings, and Chroma DB vector store.

In [17]:
# Import Path, OS, environment, NumPy, Pandas, and LangChain core components
from pathlib import Path
import getpass
import os
import shutil

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [18]:
# Verify or prompt for Google API Key configuration
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass(
        "Enter your Google API key: "
    )

print("Google API key configured successfully.")

Google API key configured successfully.


In [19]:
# Locate the target PDF document inside the data directory
DATA_DIR = Path(
    r"D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    # Automatically find the PDF if its filename is slightly different
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:")
print(PDF_PATH)

PDF found:
D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\llama2-research-paper.pdf


## 2. Document Ingestion, Metadata Enrichment, and Chunking
Load the source PDF (`llama2-research-paper.pdf`), extract page content, enrich each document page with custom domain metadata (paper section, organization, publication year), and split into chunks.

In [20]:
# Load pages from PDF file using PyPDFLoader
loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

Total PDF pages loaded: 77


In [21]:
# Inspect first page content and extracted metadata
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\Coding\\Full-Stack-GenAI-AgenticAI-Bootcamp\\05_RAG\\05_Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor K

### 2.1 Section Identification & Metadata Enrichment
Define a domain-specific section mapping function to tag each page with its major paper section (`introduction`, `pretraining`, `fine_tuning`, `safety`, `discussion`, `conclusion`, etc.).

In [22]:
# Define helper function to map PDF page numbers to paper sections
def identify_section(paper_page: int) -> str:
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"

In [23]:
# Enrich metadata for all loaded document pages
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

In [24]:
# Display enriched metadata for the first 5 pages
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\Coding\\Full-Stack-GenAI-AgenticAI-Bootcamp\\05_RAG\\05_Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/Fals

### 2.2 Text Chunking & Chunk ID Tagging
Split the loaded pages into overlapping chunks using `RecursiveCharacterTextSplitter` and assign unique, deterministic `chunk_id` metadata.

In [25]:
# Split document pages into overlapping chunks using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 77
Total chunks: 343


In [26]:
# Assign deterministic chunk_id metadata to every chunk
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

In [27]:
# Inspect first chunk content and metadata
print("Chunk content:")
print(chunks[0].page_content[:1000])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

## 3. Embeddings & Vector Store Setup (Chroma DB)
Initialize `GoogleGenerativeAIEmbeddings` using `gemini-embedding-001`, configure persistent disk storage for Chroma DB, and insert document chunks in rate-limited batches.

In [28]:
# Initialize Google Generative AI Embeddings model (gemini-embedding-001)
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [29]:
# Test embedding model output dimension
test_vector = embedding_model.embed_query(
    "What is Llama 2?"
)

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 3072
First 10 values: [-0.0013216271, -0.004591897, -0.00072388595, -0.054884955, -0.009096947, -0.01606862, -0.019948112, 0.012497553, 0.018885894, 0.011566677]


In [30]:
# Configure persistent directory path for Chroma vector store
PERSIST_DIRECTORY = DATA_DIR / "chroma_llama2_retriever"

# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

### 3.1 Batch Ingestion into Persistent Chroma DB
Populate the persistent Chroma store in small batches with retry logic for 429 rate limit errors.

In [31]:
# Initialize persistent Chroma collection
vector_store = Chroma(
    collection_name="llama2_retriever_demo",
    embedding_function=embedding_model,
    persist_directory=str(PERSIST_DIRECTORY),
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

# Add documents in batches with retry logic for rate limits
import time

batch_size = 20
total_added = 0

for i in range(0, len(chunks), batch_size):
    print(f"Adding batch {i} to {i+batch_size} of {len(chunks)}...")
    batch = chunks[i:i+batch_size]
    
    success = False
    while not success:
        try:
            added_ids = vector_store.add_documents(documents=batch)
            total_added += len(added_ids)
            success = True
            if i + batch_size < len(chunks):
                time.sleep(15)
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                print("Rate limit hit! Pausing for 60 seconds before retrying...")
                time.sleep(60)
            else:
                raise e

print("\nVector store created successfully.")
print(f"Documents added: {total_added}")
print(f"Total documents stored: {vector_store._collection.count()}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Adding batch 0 to 20 of 343...
Adding batch 20 to 40 of 343...
Adding batch 40 to 60 of 343...
Adding batch 60 to 80 of 343...
Adding batch 80 to 100 of 343...
Adding batch 100 to 120 of 343...
Adding batch 120 to 140 of 343...
Adding batch 140 to 160 of 343...
Adding batch 160 to 180 of 343...
Adding batch 180 to 200 of 343...
Adding batch 200 to 220 of 343...
Adding batch 220 to 240 of 343...
Adding batch 240 to 260 of 343...
Adding batch 260 to 280 of 343...
Adding batch 280 to 300 of 343...
Adding batch 300 to 320 of 343...
Adding batch 320 to 340 of 343...
Adding batch 340 to 360 of 343...

Vector store created successfully.
Documents added: 343
Total documents stored: 343
Persisted at: D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\chroma_llama2_retriever


### 3.2 Document Formatting Helper
Define a reusable helper function `display_documents` to clearly format and display retrieved chunks along with their metadata.

In [32]:
# Define reusable display_documents helper function
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

### 4.1 Standard Similarity Search
Perform k-Nearest Neighbor (k-NN) similarity search based on Cosine distance.

In [33]:
# Create standard Similarity retriever (k=4)
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

In [34]:
# Execute similarity search query
query = "What model sizes of Llama 2 were released?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-12
SOURCE: D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the open release of LLMs, when done safely, will be a net benefit to society. Like all LLMs,
Llama 2 is

RANK: 2
PAPER PAGE: 6
SEC

### 4.2 Maximal Marginal Relevance (MMR)
MMR optimizes for both **relevance** to the query and **diversity** among retrieved results to avoid redundant chunks. Parameter `lambda_mult` controls the balance (1.0 = pure relevance, 0.0 = pure diversity).

In [35]:
# Create Maximal Marginal Relevance (MMR) retriever (k=4, fetch_k=20, lambda_mult=0.5)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

In [36]:
# Execute MMR retriever query
query = "How was Llama 2-Chat trained and aligned?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 5
SECTION: pretraining
CHUNK ID: llama2-page-5-chunk-14
SOURCE: D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdes

RANK: 2
PAPER PAGE: 16
SEC

In [37]:
# Execute side-by-side comparison between Similarity and MMR search
query = "How was Llama 2-Chat trained and aligned?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

In [38]:
# Print side-by-side comparison of retrieved page sections
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
5 pretraining llama2-page-5-chunk-14
10 fine_tuning llama2-page-10-chunk-40
8 fine_tuning llama2-page-8-chunk-29
15 fine_tuning llama2-page-15-chunk-62

MMR results:
5 pretraining llama2-page-5-chunk-14
16 fine_tuning llama2-page-16-chunk-68
22 safety llama2-page-22-chunk-97
10 fine_tuning llama2-page-10-chunk-36


### 4.3 Similarity Score Thresholding
Filter out irrelevant results by requiring a minimum similarity score threshold (e.g., `score_threshold = 0.64`).

In [39]:
# Create similarity score threshold retriever (k=10, score_threshold=0.64)
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.64,
    }
)

In [40]:
# Execute score threshold retriever query
query = "What safety techniques were used for Llama 2-Chat?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 27
SECTION: safety
CHUNK ID: llama2-page-27-chunk-119
SOURCE: D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
4.2.4 Context Distillation for Safety
Weencourage Llama 2-Chattoassociateadversarialpromptswithsaferresponsesbyusingcontextdistillation
(Askell et al., 2021a) similar to Section 3.3. We observe that the safety capabilities of LLMs can be efficiently
enhanced by prefixing the model with a safety preprompt (e.g.,“You are a safe and responsible assistant”).
Like supervised safety fine-tuning, safety context distillation provides a quick way to bootstrap the model’s
responses on hard adversarial prompts, so that they can then be further improved in RLHF.
Specifically, we apply context distillation by prefixing a safety preprompt to adversarial prompts to generate
safer responses, and then fine-t

RANK: 2
PAPER PAGE: 24
SECTI

In [41]:
# Search with relevance scores using similarity_search_with_relevance_scores
query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.7897
Paper page: 27
Section: safety
4.2.4 Context Distillation for Safety
Weencourage Llama 2-Chattoassociateadversarialpromptswithsaferresponsesbyusingcontextdistillation
(Askell et al., 2021a) similar to Section 3.3. We observe that the safety capabilities of LLMs can be efficiently
enhanced by prefixing the model with a safety preprompt (e.g.,“You are a safe and responsible assistant”).
Like supervised safety fine-tuning, safety context distillation provides a quick way to bootstrap the model’s
responses on hard adversarial pro
Rank: 2
Relevance score: 0.7883
Paper page: 24
Section: safety
4.2.2 Safety Supervised Fine-Tuning
In accordance with the established guidelines from Section 4.2.1, we gather prompts and demonstrations
of safe model responses from trained annotators, and use the data for supervised fine-tuning in the same
manner as described in Section 3.1. An example can be found in Table 5.
The annotators are instructed to initially come up with p

## 5. Mathematical Foundations of Distance Metrics
Inspect the raw high-dimensional vector representations and compare **Cosine Similarity**, **Euclidean Distance**, and **Dot Product** mathematically.

In [42]:
# Retrieve candidate documents for distance metric comparison
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


In [44]:
# Generate raw NumPy vector arrays for query and document candidates
query_vector = np.asarray(
    embedding_model.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embedding_model.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (3072,)
Document-vectors shape: (6, 3072)


In [45]:
# Define manual functions for Cosine Similarity, Euclidean Distance, and Dot Product
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

In [46]:
# Compute distance metrics for candidate documents and build metric table
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,32,discussion,llama2-page-32-chunk-138,0.746502,0.712037,0.746502,"bility, seemed a somewhat shadowy field for th..."
1,10,fine_tuning,llama2-page-10-chunk-35,0.739230,0.722177,0.739230,"sampled human preferences, whereby human annot..."
2,15,fine_tuning,llama2-page-15-chunk-60,0.733581,0.729958,0.733581,"regression in some capabilities. For example, ..."
3,17,fine_tuning,llama2-page-17-chunk-73,0.726230,0.739959,0.726231,system message during the conversation by inte...
4,24,safety,llama2-page-24-chunk-105,0.720886,0.747146,0.720886,"explain why the topic might be sensitive, and ..."
5,32,discussion,llama2-page-32-chunk-139,0.720598,0.747532,0.720598,distribution and aligns towards the human pref...


In [47]:
# Display ranking sorted by Cosine Similarity
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,32,discussion,0.746502,"bility, seemed a somewhat shadowy field for th..."
1,10,fine_tuning,0.739230,"sampled human preferences, whereby human annot..."
2,15,fine_tuning,0.733581,"regression in some capabilities. For example, ..."
3,17,fine_tuning,0.726230,system message during the conversation by inte...
4,24,safety,0.720886,"explain why the topic might be sensitive, and ..."
5,32,discussion,0.720598,distribution and aligns towards the human pref...


In [48]:
# Display ranking sorted by Euclidean Distance
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,32,discussion,0.712037,"bility, seemed a somewhat shadowy field for th..."
1,10,fine_tuning,0.722177,"sampled human preferences, whereby human annot..."
2,15,fine_tuning,0.729958,"regression in some capabilities. For example, ..."
3,17,fine_tuning,0.739959,system message during the conversation by inte...
4,24,safety,0.747146,"explain why the topic might be sensitive, and ..."
5,32,discussion,0.747532,distribution and aligns towards the human pref...


In [49]:
# Display ranking sorted by Dot Product
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,32,discussion,0.746502,"bility, seemed a somewhat shadowy field for th..."
1,10,fine_tuning,0.739230,"sampled human preferences, whereby human annot..."
2,15,fine_tuning,0.733581,"regression in some capabilities. For example, ..."
3,17,fine_tuning,0.726231,system message during the conversation by inte...
4,24,safety,0.720886,"explain why the topic might be sensitive, and ..."
5,32,discussion,0.720598,distribution and aligns towards the human pref...


### 5.1 Equivalence of Normalized Dot Product & Cosine Similarity
Demonstrate mathematically and programmatically that when vectors are L2-normalized, Dot Product is identical to Cosine Similarity.

In [50]:
# Prove equivalence of normalized Dot Product and Cosine Similarity programmatically
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.74650203 0.73923043 0.73358102 0.7262304  0.72088632 0.72059825]

Dot product after normalization:
[0.74650203 0.73923043 0.73358102 0.7262304  0.72088632 0.72059825]

Are they approximately equal? True


### 6.1 Single-Field Pre-Filtering
Restrict vector search strictly to document chunks where `section == 'fine_tuning'`.

In [51]:
# Create retriever with section pre-filter (section == 'fine_tuning')
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "fine_tuning"
        },
    }
)

In [52]:
# Execute pre-filtered retriever query
query = "How was Llama 2-Chat aligned with human preferences?"

fine_tuning_documents = fine_tuning_retriever.invoke(query)

display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-36
SOURCE: D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
tworesponsestoagivenpromptaresampledfromtwodifferentmodelvariants,andvaryingthetemperature
hyper-parameter. Inadditiontogivingparticipantsaforcedchoice,wealsoaskannotatorstolabelthedegree
to which they prefer their chosen response over the alternative: either their choice issignificantly better, better,
slightly better, ornegligibly better/ unsure.
For our collection of preference annotations, we focus on helpfulness and safety. Helpfulness refers to how
well Llama 2-Chat responses fulfill users’ requests and provide requested information; safety refers to
whether Llama 2-Chat’s responses are unsafe, e.g.,“giving detailed instructions on making a bomb”could
be considered helpful but is unsaf

RANK: 2
PAPER PAGE: 10
S

In [53]:
# Validate that all returned documents match the fine_tuning section filter
for document in fine_tuning_documents:
    assert document.metadata["section"] == "fine_tuning"

print("All returned documents are from the fine_tuning section.")

All returned documents are from the fine_tuning section.


### 6.2 Multi-Field Compound Pre-Filtering
Apply boolean logical operators (`$and`, `$eq`) to filter across multiple metadata fields simultaneously (`section`, `year`, `organization`).

In [54]:
# Create retriever with multi-field compound pre-filter ($and operator)
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        "$eq": "fine_tuning"
                    }
                },
                {
                    "year": {
                        "$eq": 2023
                    }
                },
                {
                    "organization": {
                        "$eq": "Meta"
                    }
                },
            ]
        },
    }
)

In [55]:
# Execute compound pre-filtered retriever query
query = "How was human preference data collected?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-35
SOURCE: D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
sampled human preferences, whereby human annotators select which of two model outputs they prefer.
This human feedback is subsequently used to train a reward model, which learns patterns in the preferences
of the human annotators and can then automate preference decisions.
3.2.1 Human Preference Data Collection
Next, we collect human preference data for reward modeling. We chose a binary comparison protocol over
other schemes, mainly because it enables us to maximize the diversity of collected prompts. Still, other
strategies are worth considering, which we leave for future work.
Our annotation procedure proceeds as follows. We ask annotators to first write a prompt, then choose
between two 

RANK: 2
PAPER PAGE: 10
S

In [56]:
# Execute pre-filtered query on reward model training
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "fine_tuning"
            }
        },
        {
            "year": {
                "$eq": 2023
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "How was the reward model trained?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 15
SECTION: fine_tuning
CHUNK ID: llama2-page-15-chunk-63
SOURCE: D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
We iteratively improve the policy by sampling promptsp from our datasetD and generationsg from the
policy π and use the PPO algorithm and loss function to achieve this objective.
The final reward function we use during optimization,
R(g | p) = ˜Rc(g | p) − βDKL (πθ(g | p) ∥ π0(g | p)) (4)
contains a penalty term for diverging from the original policyπ0. As was observed in other works (Stiennon
et al., 2020; Ouyang et al., 2022), we find this constraint is useful for training stability, and to reduce reward
hackingwherebywewouldachievehighscoresfromtherewardmodelbutlowscoresfromhumanevaluation.
We defineRc to be a piecewise combination of the safety (Rs) and helpfulness (Rh) reward models. We

RANK: 2
PAPER PAGE: 11
S

### 6.3 Pre-Filtering vs. Post-Filtering Comparison
Demonstrate why native Pre-Filtering is superior: Post-Filtering over-fetches candidates and discards non-matching items, risking empty or incomplete results if `k` is small.

In [57]:
# Retrieve un-filtered candidates for post-filtering comparison
unfiltered_candidates = vector_store.similarity_search(
    query="How was the reward model trained?",
    k=15,
)

In [58]:
# Apply Python post-filtering on un-filtered candidates
post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 15
SECTION: fine_tuning
CHUNK ID: llama2-page-15-chunk-63
SOURCE: D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\05_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
We iteratively improve the policy by sampling promptsp from our datasetD and generationsg from the
policy π and use the PPO algorithm and loss function to achieve this objective.
The final reward function we use during optimization,
R(g | p) = ˜Rc(g | p) − βDKL (πθ(g | p) ∥ π0(g | p)) (4)
contains a penalty term for diverging from the original policyπ0. As was observed in other works (Stiennon
et al., 2020; Ouyang et al., 2022), we find this constraint is useful for training stability, and to reduce reward
hackingwherebywewouldachievehighscoresfromtherewardmodelbutlowscoresfromhumanevaluation.
We defineRc to be a piecewise combination of the safety (Rs) and helpfulness (Rh) reward models. We

RANK: 2
PAPER PAGE: 11
S

In [59]:
# Compare count of documents retrieved pre-filtering vs. post-filtering
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 14


## 7. Hybrid Search (Sparse BM25 + Dense Vector + Reciprocal Rank Fusion)
Combine keyword-based sparse retrieval (BM25) with semantic dense vector retrieval to capture both exact terminology/acronyms and semantic context.

In [ ]:
# Import BM25, EnsembleRetriever, CrossEncoder, and LLM modules
import os
from typing import List

from pydantic import BaseModel, Field

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [ ]:
# Redefine display_documents helper with custom title support
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

In [ ]:
# Helper function to deduplicate retrieved document chunks preserving order
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

### 7.1 Sparse Keyword Retrieval (BM25)
Initialize a `BM25Retriever` to handle keyword searches, exact product names, and technical terms.

In [ ]:
# Initialize BM25 keyword retriever from document chunks
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(chunks)

# Final number of results
bm25_retriever.k = 4

In [ ]:
# Test BM25 sparse keyword retriever on technical terms query
sparse_query = "Grouped-Query Attention GQA 70B"

sparse_documents = bm25_retriever.invoke(sparse_query)

display_documents(
    sparse_documents,
    title="Sparse Retrieval: BM25 Results",
)

### 7.2 Dense Vector Retrieval Comparison
Compare BM25 sparse results against Dense vector search on technical queries.

In [ ]:
# Initialize Dense Vector retriever (k=4)
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

In [ ]:
# Test Dense Vector retriever on inference scalability query
dense_query = (
    "How did Meta improve inference scalability "
    "for the largest Llama 2 models?"
)

dense_documents = dense_retriever.invoke(dense_query)

display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)

In [ ]:
# Compare BM25 vs Dense Vector results on a query containing both keywords and semantic intent
comparison_query = (
    "How did grouped-query attention improve "
    "Llama 2 inference scalability?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)

In [ ]:
# Print side-by-side rankings of Sparse vs Dense results
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

### 7.3 Hybrid Ensemble Search (Weighted RRF)
Combine BM25 and Dense retrievers into a single `EnsembleRetriever` using Reciprocal Rank Fusion (RRF) to merge candidate rankings.

In [ ]:
# Configure candidate count for hybrid search
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 8
    },
)

In [ ]:
# Create EnsembleRetriever combining BM25 and Dense Vector search with equal RRF weights (0.5 / 0.5)
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever,
    ],
    weights=[
        0.5,  # BM25 weight
        0.5,  # Dense-retrieval weight
    ],
)

In [ ]:
# Test Hybrid Ensemble retriever query
hybrid_query = (
    "Llama 2 70B grouped-query attention "
    "and inference scalability"
)

hybrid_documents = hybrid_retriever.invoke(hybrid_query)

display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense + RRF",
    max_documents=6,
)

### Hybrid Retrieval Architecture (BM25 + Dense RRF)

```text
                         ┌── BM25 Retriever (Keywords) ────┐
User Query ──────────────┤                                  ├── Weighted Reciprocal Rank Fusion (RRF) ──► Combined Ranking
                         └── Dense Retriever (Semantic) ────┘
```

**How Reciprocal Rank Fusion (RRF) Works:**
RRF calculates a score for each document based on its rank in each individual retriever:
$$\text{RRF\_Score}(d) = \sum_{r \in R} \frac{w_r}{k + \text{rank}_r(d)}$$
where $w_r$ is the weight of retriever $r$, and $k$ is a constant (typically 60).

In [ ]:
# Compare Sparse, Dense, and Hybrid retrieval results on a complex query
test_query = (
    "How did Meta make Llama 2 70B efficient "
    "for large-scale inference?"
)

sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)

## 8. Query Transformation Techniques
Use LLM intelligence to rephrase, expand, or break down user queries before sending them to retrievers.

In [ ]:
# Initialize ChatOpenAI model for query transformations
CHAT_MODEL = os.environ.get(
    "OPENAI_CHAT_MODEL",
    "gpt-4.1-mini",
)

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)

### 8.1 Conversational Query Rewriting
Transform ambiguous conversational follow-up questions (e.g. *"What did Meta do after that?"*) into standalone, self-contained search queries using chat history.

In [ ]:
# Construct ChatPromptTemplate for conversational query rewriting
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.
""",
        ),
        (
            "human",
            """
Conversation history:
{chat_history}

Current query:
{query}
""",
        ),
    ]
)

In [ ]:
# Build query rewriting chain using LCEL
query_rewriting_chain = (
    query_rewriting_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Execute query rewriting chain on a conversational follow-up question
chat_history = """
User: How was Llama 2-Chat initially fine-tuned?
Assistant: It first underwent supervised fine-tuning.
"""

original_query = "What did Meta do after that?"

rewritten_query = query_rewriting_chain.invoke(
    {
        "chat_history": chat_history,
        "query": original_query,
    }
).strip()

print("Original query:")
print(original_query)

print("\nRewritten query:")
print(rewritten_query)

In [ ]:
# Execute hybrid retrieval using the rewritten query
rewritten_query_documents = hybrid_retriever.invoke(
    rewritten_query
)

display_documents(
    rewritten_query_documents,
    title="Documents Retrieved Using the Rewritten Query",
    max_documents=5,
)

### 8.2 Query Expansion & Multi-Query Retrieval
Generate multiple synthetic variations of a user query using structured output, execute searches across all variations, and merge/deduplicate candidate documents.

In [ ]:
# Define Pydantic schema for Query Expansion output
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

In [ ]:
# Wrap ChatOpenAI with structured output for Query Expansion
query_expansion_llm = llm.with_structured_output(
    ExpandedQueryOutput
)

In [ ]:
# Create ChatPromptTemplate for generating alternative search query variations
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

In [ ]:
# Build query expansion chain
query_expansion_chain = (
    query_expansion_prompt
    | query_expansion_llm
)

In [ ]:
# Execute query expansion chain to generate 4 alternative query formulations
original_query = (
    "How was Llama 2-Chat improved using human feedback?"
)

expanded_output = query_expansion_chain.invoke(
    {
        "query": original_query
    }
)

all_expanded_queries = [
    original_query,
    *expanded_output.queries,
]

print("Generated search queries:\n")

for number, query in enumerate(
    all_expanded_queries,
    start=1,
):
    print(f"{number}. {query}")

In [ ]:
# Retrieve candidates across all expanded queries and deduplicate results
expanded_query_documents = []

for query in all_expanded_queries:
    current_documents = dense_retriever.invoke(query)
    expanded_query_documents.extend(current_documents)

expanded_query_documents = deduplicate_documents(
    expanded_query_documents
)

display_documents(
    expanded_query_documents,
    title="Query Expansion: Combined Unique Documents",
    max_documents=10,
)

In [ ]:
# Execute query expansion using LangChain's built-in MultiQueryRetriever
from langchain_classic.retrievers.multi_query import (
    MultiQueryRetriever,
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm,
    include_original=True,
)

multi_query_documents = multi_query_retriever.invoke(
    "How did human feedback improve Llama 2-Chat?"
)

display_documents(
    multi_query_documents,
    title="Built-in MultiQueryRetriever Results",
    max_documents=10,
)

### 8.3 Query Decomposition for Complex Questions
Break multi-part, complex questions into independent atomic sub-queries, retrieve context for each sub-query, and consolidate the evidence.

In [ ]:
# Define Pydantic schema for Query Decomposition output
class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description=(
            "Independent and atomic search queries required "
            "to answer the complete user question."
        )
    )

In [ ]:
# Wrap ChatOpenAI with structured output for Query Decomposition
query_decomposition_llm = llm.with_structured_output(
    DecomposedQueryOutput
)

In [ ]:
# Create ChatPromptTemplate for breaking complex questions into atomic sub-queries
query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Break the user's complex question into independent,
atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.
""",
        ),
        (
            "human",
            "Complex query: {query}",
        ),
    ]
)

In [ ]:
# Build query decomposition chain
query_decomposition_chain = (
    query_decomposition_prompt
    | query_decomposition_llm
)

In [ ]:
# Execute query decomposition on a complex multi-part question
complex_query = """
Compare Llama 2 pretraining with Llama 2-Chat fine-tuning,
and explain how Meta improved model safety.
"""

decomposed_output = query_decomposition_chain.invoke(
    {
        "query": complex_query
    }
)

sub_queries = decomposed_output.sub_queries

print("Original complex query:")
print(complex_query)

print("\nGenerated sub-queries:")

for number, query in enumerate(sub_queries, start=1):
    print(f"{number}. {query}")

In [ ]:
# Execute hybrid retrieval for each generated sub-query
decomposition_results = {}

for sub_query in sub_queries:
    decomposition_results[sub_query] = hybrid_retriever.invoke(
        sub_query
    )

In [ ]:
# Display documents retrieved for each sub-query independently
for sub_query, documents in decomposition_results.items():
    display_documents(
        documents,
        title=f"Sub-query: {sub_query}",
        max_documents=4,
    )

In [ ]:
# Consolidate and deduplicate evidence from all sub-queries
all_decomposition_documents = []

for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents)

all_decomposition_documents = deduplicate_documents(
    all_decomposition_documents
)

display_documents(
    all_decomposition_documents,
    title="Combined Evidence from All Sub-Queries",
    max_documents=12,
)

### Query Decomposition Architecture

```text
Complex Query ──► LLM Decomposition ──► Atomic Sub-queries ──► Parallel Retrieval ──► Merge Evidence ──► Deduplicate ──► Final Context
```

## 9. Contextual Compression & Cross-Encoder Reranking
Use a two-stage retrieval pipeline: retrieve a large candidate pool (e.g. top 20) with fast vector search, then re-rank candidates using a precise Cross-Encoder model (`ms-marco-MiniLM-L6-v2`).

In [ ]:
# Configure candidate vector retriever to fetch top 20 candidates for reranking
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 20
    },
)

### Two-Stage Reranking Pipeline

```text
User Query ──► Fast Vector Search ──► Top 20 Candidates ──► Cross-Encoder Reranker ──► Top 5 Final Context
```

### 9.1 Dense Candidate Reranking
Rerank top 20 dense vector candidates to get the top 5 most relevant documents.

In [ ]:
# Initialize HuggingFace Cross-Encoder model (ms-marco-MiniLM-L6-v2)
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
)

In [ ]:
# Create CrossEncoderReranker targeting top 5 final documents
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

In [ ]:
# Build ContextualCompressionRetriever wrapping candidate retriever and reranker
reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [ ]:
# Define test query for reranking
reranking_query = (
    "How did Meta collect and use human preference data "
    "to train Llama 2-Chat?"
)

In [ ]:
# Display initial top 10 candidates before reranking
initial_candidates = candidate_retriever.invoke(
    reranking_query
)

display_documents(
    initial_candidates,
    title="Before Reranking: Initial Vector Candidates",
    max_documents=10,
)

In [ ]:
# Display final top 5 reranked documents
reranked_documents = reranking_retriever.invoke(
    reranking_query
)

display_documents(
    reranked_documents,
    title="After Reranking: Final Top Documents",
    max_documents=5,
)

### Cross-Encoder Reranking Architecture

Unlike Bi-Encoders (which compute query and document embeddings independently), a **Cross-Encoder** processes the `(query, document)` pair simultaneously through full cross-attention layers, yielding significantly higher relevance accuracy.

```text
User Query ────┐
               ├──► Dense Vector Search ──► Top 20 Candidates ──► Cross-Encoder Reranker ──► Top 5 Documents
Document Pool ─┘
```

### 9.2 Advanced Hybrid + Cross-Encoder Reranking Pipeline
Combine **Hybrid Search (BM25 + Dense RRF)** with **Cross-Encoder Reranking** for state-of-the-art RAG retrieval performance.

In [ ]:
# Configure hybrid candidate retriever (BM25 + Dense) fetching top 15 candidates each
bm25_retriever.k = 15

dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 15
    },
)

hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_candidate_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

In [ ]:
# Build ContextualCompressionRetriever wrapping Hybrid Ensemble candidate retriever and CrossEncoder reranker
hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [ ]:
# Execute ultimate Hybrid + CrossEncoder reranking pipeline query
query = (
    "What techniques did Meta use to improve "
    "the helpfulness and safety of Llama 2-Chat?"
)

final_documents = hybrid_reranking_retriever.invoke(query)

display_documents(
    final_documents,
    title="Hybrid Retrieval + Cross-Encoder Reranking",
    max_documents=5,
)

### Ultimate RAG Pipeline: Hybrid Search + Cross-Encoder Reranking

```text
                         ┌── BM25 Search (Exact Keywords) ──┐
User Query ──────────────┤                                  ├──► Weighted RRF ──► Candidate Pool ──► Cross-Encoder Reranker ──► Final Top Docs
                         └── Dense Search (Semantic Match) ─┘
```

## 10. Masterclass Summary & Method Selection Guide
Quick reference guide summarizing when to use each retrieval strategy in production RAG systems.

## Summary of Advanced Retrieval Strategies

| Technique | Method / Class | Primary Use Case | Key Advantage |
|---|---|---|---|
| **Sparse Retrieval** | `BM25Retriever` | Exact keyword matches, part numbers, technical codes | Captures exact lexical tokens |
| **Dense Retrieval** | `VectorStoreRetriever` | Conceptual & semantic similarity search | Captures intent and meaning |
| **Hybrid Retrieval** | `EnsembleRetriever` (BM25 + Dense) | General-purpose high-performance search | Combines lexical precision + semantic depth |
| **Query Rewriting** | LLM Chain (`ChatPromptTemplate`) | Conversational chat history resolution | Converts incomplete questions into standalone queries |
| **Query Expansion** | `MultiQueryRetriever` / LLM | Overcoming vocabulary mismatch & single-phrasing bias | Generates 4+ paraphrased search queries |
| **Query Decomposition**| LLM Chain (`BaseModel` output) | Multi-part or comparative questions | Splits complex questions into atomic sub-searches |
| **Cross-Encoder Reranking** | `CrossEncoderReranker` | Precision re-ordering of top candidate chunks | Uses joint query-document cross-attention for top accuracy |